DATASET LOADING

In [ ]:
!git clone -q https://github.com/harish-rbuilds/Canteen_DWDM.git
import pandas as pd
import sqlite3
from pathlib import Path
BASE = Path("Canteen_DWDM/raw_data")
students = pd.read_csv(BASE / "students.csv")
menu = pd.read_csv(BASE / "menu_items.csv")
canteens = pd.read_csv(BASE / "canteens.csv")
transactions = pd.read_csv(BASE / "transactions.csv")
calendar = pd.read_csv(BASE / "calendar.csv")
items = pd.read_csv(BASE / "raw_transaction_items.csv")
print("Dataset Shapes:")
print({
"students": students.shape,
"menu": menu.shape,
"canteens": canteens.shape,
"transactions": transactions.shape,
"calendar": calendar.shape,
"items": items.shape
})

fatal: destination path 'Canteen_DWDM' already exists and is not an empty directory.
Dataset Shapes:
{'students': (2000, 6), 'menu': (50, 8), 'canteens': (4, 5), 'transactions': (35000, 10), 'calendar': (365, 13), 'items': (83283, 6)}


DATA INSPECTION

In [ ]:
print("\nMissing Values:")
print(items.isna().sum())
print("\nDuplicate Business Keys:",
items.duplicated(["Order_ID", "Menu_Item_ID"]).sum())
print("\nData Types:")
print(items.dtypes)


Missing Values:
Order_ID              0
Menu_Item_ID          0
Quantity              0
Unit_Price            0
Discount_Percent    100
Total_Amount        101
dtype: int64

Duplicate Business Keys: 80

Data Types:
Order_ID             object
Menu_Item_ID         object
Quantity              int64
Unit_Price            int64
Discount_Percent    float64
Total_Amount        float64
dtype: object


DATA PREPROCESSING

In [ ]:
clean = items.copy()
# Standardize identifiers
for c in ["Order_ID", "Menu_Item_ID"]:
    clean[c] = clean[c].astype(str).str.strip()
# Convert numerical columns
num_cols = [
"Quantity",
"Unit_Price",
"Discount_Percent",
"Total_Amount"
]
clean[num_cols] = clean[num_cols].apply(
pd.to_numeric,
errors="coerce"
)
# Missing discount → 0%
clean["Discount_Percent"] = (
clean["Discount_Percent"].fillna(0)
)
# Recalculate total amount
clean["Total_Amount"] = (
clean["Quantity"]
* clean["Unit_Price"]
* (1 - clean["Discount_Percent"] / 100)
)
# Remove duplicate business keys
clean = clean.drop_duplicates(
["Order_ID", "Menu_Item_ID"]
)
# Validate basic ranges
clean = clean[
(clean["Quantity"] > 0) &
(clean["Unit_Price"] >= 0) &
(clean["Discount_Percent"].between(0, 100))
].copy()
clean["Total_Amount"] = clean["Total_Amount"].round(2)
print("\nClean Dataset Shape:", clean.shape)


Clean Dataset Shape: (83203, 6)


DATA VALIDATION

In [ ]:
transactions["Order_Date"] = pd.to_datetime(
transactions["Order_Date"],
errors="coerce"
)
transactions["Order_Time"] = pd.to_datetime(
transactions["Order_Time"],
format="%H:%M:%S",
errors="coerce"
).dt.time
checks = {
"missing_clean_values":
int(clean.isna().sum().sum()),
"duplicate_business_keys":
int(clean.duplicated(
["Order_ID", "Menu_Item_ID"]
).sum()),
"invalid_student_ids":
int((~transactions["Student_ID"].isin(
students["Student_ID"]
)).sum()),
"invalid_canteen_ids":
int((~transactions["Canteen_ID"].isin(
canteens["Canteen_ID"]
)).sum()),
"invalid_menu_ids":
int((~clean["Menu_Item_ID"].isin(
menu["Menu_Item_ID"].astype(str)
)).sum())
}
print("\nValidation Results:")
print(pd.Series(checks))
expected = (
clean["Quantity"]
* clean["Unit_Price"]
* (1 - clean["Discount_Percent"] / 100)
).round(2)
print(
"Amount mismatches:",
int((clean["Total_Amount"] != expected).sum())
)


Validation Results:
missing_clean_values       0
duplicate_business_keys    0
invalid_student_ids        0
invalid_canteen_ids        0
invalid_menu_ids           0
dtype: int64
Amount mismatches: 0


SAVE CLEAN DATASET

In [ ]:
OUT = Path("Canteen_DWDM/preprocessing")
OUT.mkdir(parents=True, exist_ok=True)
clean.to_csv(
OUT / "clean_transaction_items.csv",
index=False
)
print("\nClean dataset saved successfully.")


Clean dataset saved successfully.


DATA WAREHOUSE DEVELOPMENT

In [ ]:
#create database
DB = Path(
"Canteen_DWDM/data_warehouse/canteen_dw.db"
)
DB.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB)
#create dimensions
# Date Dimension
dim_date = calendar.copy()
dim_date["Date"] = pd.to_datetime(
dim_date["Date"]
).dt.strftime("%Y-%m-%d")
dim_date.to_sql(
"Dim_Date",
conn,
if_exists="replace",
index=False
)
# Time Dimension
dim_time = (
transactions[["Order_Time"]]
.drop_duplicates()
.dropna()
.copy()
)
dim_time["Time_ID"] = range(
1,
len(dim_time) + 1
)
dim_time.to_sql(
"Dim_Time",
conn,
if_exists="replace",
index=False
)
# Student Dimension
students.to_sql(
"Dim_Student",
conn,
if_exists="replace",
index=False
)
# Menu Item Dimension
menu.to_sql(
"Dim_Menu_Item",
conn,
if_exists="replace",
index=False
)
# Canteen Dimension
canteens.to_sql(
"Dim_Canteen",
conn,
if_exists="replace",
index=False
)
# Payment Dimension
dim_payment = (
transactions[["Payment_Method"]]
.drop_duplicates()
.reset_index(drop=True)
)
dim_payment["Payment_ID"] = (
dim_payment.index + 1
)
dim_payment.to_sql(
"Dim_Payment",
conn,
if_exists="replace",
index=False
)
print("\nDimension tables created.")
#create fact_table
t = transactions.copy()
t["Order_ID"] = t["Order_ID"].astype(str)
clean["Order_ID"] = clean["Order_ID"].astype(str)
clean["Menu_Item_ID"] = clean["Menu_Item_ID"].astype(str)
# Combine transaction information with item-level sales
fact = clean.merge(
t[[
"Order_ID",
"Student_ID",
"Order_Date",
"Order_Time",
"Canteen_ID",
"Payment_Method"
]],
on="Order_ID",
how="left"
)
# Add payment key
fact = fact.merge(
dim_payment,
on="Payment_Method",
how="left"
)
# Date key
fact["Date_ID"] = (
fact["Order_Date"].astype(str)
)
# Time key
time_map = dim_time.set_index(
dim_time["Order_Time"].astype(str)
)["Time_ID"].to_dict()

fact["Time_ID"] = (
fact["Order_Time"]
.astype(str)
.map(time_map)
)
# Select fact table columns
fact = fact[[
"Date_ID",
"Time_ID",
"Student_ID",
"Menu_Item_ID",
"Canteen_ID",
"Payment_ID",
"Order_ID",
"Quantity",
"Unit_Price",
"Discount_Percent",
"Total_Amount"
]]
# Rename measure
fact = fact.rename(
columns={
"Total_Amount": "Sales_Amount"
}
)
# Load fact table
fact.to_sql(
"Fact_Canteen_Sales",
conn,
if_exists="replace",
index=False
)
print("Fact table created.")
print("Fact rows:", len(fact))


Dimension tables created.
Fact table created.
Fact rows: 83203


OLAP OPERATIONS

In [ ]:
#roll up
# Day → Month → Year
rollup = pd.read_sql(
"""
SELECT
d.Year,
d.Month,
SUM(f.Sales_Amount) AS Sales
FROM Fact_Canteen_Sales f
JOIN Dim_Date d
ON f.Date_ID = CAST(d.Date AS TEXT)
GROUP BY
d.Year,
d.Month
ORDER BY
d.Year,
d.Month
""",
conn
)
print("\nROLL-UP:")
display(rollup.head())


ROLL-UP:


,Year,Month,Sales
0,2026,1,446705.50
1,2026,2,417867.25
2,2026,3,452420.25
3,2026,4,444451.50
4,2026,5,456047.00


In [ ]:
#drill down
# Year → Month → Day
drilldown = pd.read_sql(
"""
SELECT
d.Year,
d.Month,
d.Day,
SUM(f.Sales_Amount) AS Sales
FROM Fact_Canteen_Sales f
JOIN Dim_Date d
ON f.Date_ID = CAST(d.Date AS TEXT)
GROUP BY
d.Year,
d.Month,
d.Day
ORDER BY
d.Year,
d.Month,
d.Day
""",
conn
)

print("\nDRILL-DOWN:")
display(drilldown.head())




DRILL-DOWN:


,Year,Month,Day,Sales
0,2026,1,1,14201.75
1,2026,1,2,16411.75
2,2026,1,3,17175.25
3,2026,1,4,14620.25
4,2026,1,5,12559.50


In [ ]:
#slice
# Select one canteen
slice_c01 = pd.read_sql(
"""
SELECT
c.Canteen_Name,
SUM(f.Sales_Amount) AS Sales
FROM Fact_Canteen_Sales f
JOIN Dim_Canteen c
ON f.Canteen_ID = c.Canteen_ID
WHERE f.Canteen_ID = 'C01'
GROUP BY
c.Canteen_Name
""",
conn
)
print("\nSLICE - CANTEEN C01:")
display(slice_c01)


SLICE - CANTEEN C01:


,Canteen_Name,Sales
0,Central Canteen,2131239.0


In [ ]:
#dice
# Months 1 and 2
# Canteens C01 and C02
# Payment = UPI
dice = pd.read_sql(
"""
SELECT
d.Month,
f.Canteen_ID,
p.Payment_Method,
SUM(f.Sales_Amount) AS Sales
FROM Fact_Canteen_Sales f
JOIN Dim_Date d
ON f.Date_ID = CAST(d.Date AS TEXT)
JOIN Dim_Payment p
ON f.Payment_ID = p.Payment_ID
WHERE d.Month IN (1, 2)
AND f.Canteen_ID IN ('C01', 'C02')
AND p.Payment_Method = 'UPI'
GROUP BY
d.Month,
f.Canteen_ID,
p.Payment_Method
""",
conn
)
print("\nDICE:")
display(dice)


DICE:


,Month,Canteen_ID,Payment_Method,Sales
0,1,C01,UPI,109507.50
1,1,C02,UPI,48612.00
2,2,C01,UPI,103237.25
3,2,C02,UPI,46217.75


In [ ]:
#pivot
# Category × Month
pivot = pd.read_sql(
"""
SELECT
m.Category,
d.Month,
SUM(f.Sales_Amount) AS Sales
FROM Fact_Canteen_Sales f
JOIN Dim_Menu_Item m
ON f.Menu_Item_ID = m.Menu_Item_ID
JOIN Dim_Date d
ON f.Date_ID = CAST(d.Date AS TEXT)
GROUP BY
m.Category,
d.Month
ORDER BY
m.Category,
d.Month
""",
conn
).pivot(
index="Category",
columns="Month",
values="Sales"
)
print("\nPIVOT - CATEGORY × MONTH:")
display(pivot)



PIVOT - CATEGORY × MONTH:


Month,1,2,3,4,5,6,7,8,9,10,11,12
Category,,,,,,,,,,,,
Bakery,14045.00,11962.75,13655.00,12847.25,14474.50,13556.50,13162.0,13948.75,12749.50,14078.50,12984.50,13074.25
Beverage,68001.75,66604.25,70928.50,69070.00,70420.50,65171.00,69670.5,71828.25,64858.75,67438.50,70696.00,67422.25
Breakfast,47493.25,42637.25,48856.75,46850.25,49069.25,47571.75,46599.5,47242.00,45005.50,47251.25,46542.00,47875.50
Dessert,4617.25,4037.00,4749.25,4559.50,5508.25,4306.50,4361.5,5065.50,4408.25,4611.75,4507.25,4515.50
Fast Food,102158.25,91860.75,100347.50,92977.50,103517.75,93133.75,96298.5,103064.25,89281.25,100116.25,95373.25,93908.25
Healthy,8872.75,7198.00,9033.25,8460.25,8919.50,8892.25,8526.0,8575.75,7708.75,8617.75,8529.75,8230.25
Main Course,158366.25,152579.25,158549.75,166315.50,159500.25,153970.00,168108.0,148603.50,159640.00,157475.25,156778.75,161339.50
Snack,43151.00,40988.00,46300.25,43371.25,44637.00,42920.75,42622.0,45994.50,40812.50,42256.25,43055.75,44272.25


In [ ]:
#close database
conn.close()
print("Database connection closed.")

Database connection closed.
